# Fashion-MNIST — Tam Deney Pipeline (Güvenli + Sunum Çıktıları)

**Sıra:** Setup → Baseline (PixelCNN++) → ARPG → K-sweep → Rapor grafikleri → Drive yedek

**Güvenlik:** Her 5 epoch'ta checkpoint + `last.pt` + `best.pt` + Drive yedek

**Kopma sonrası:** Runtime yeniden açınca aynı notebook'ta train cell'ini tekrar çalıştır → otomatik resume

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/comp547_outputs/fashion_mnist'
!mkdir -p "{DRIVE_ROOT}"

In [ ]:
%cd /content
!rm -rf COMP547PROJECT
!git clone https://github.com/oaydogdu/COMP547PROJECT.git
%cd COMP547PROJECT
!git pull origin main
!pip install -q -r requirements.txt

In [ ]:
import sys, torch
from pathlib import Path
sys.path.insert(0, 'src')
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    torch.cuda.empty_cache()
# import smoke test
from KlassikAR.pixelcnnpp_runner import train_pixelcnnpp
from ARPG.arpg_runner import train_arpg
print('Imports OK')

In [ ]:
# Opsiyonel: önceki Drive yedeğinden checkpoint geri yükle (runtime sıfırlandıysa)
import shutil
from pathlib import Path
DRIVE_ROOT = '/content/drive/MyDrive/comp547_outputs/fashion_mnist'
for name in ['pixelcnnpp_fashion_e20', 'arpg_fashion']:
    src = Path(DRIVE_ROOT) / name
    dst = Path('results') / name
    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print('restored', name)
    else:
        print('no drive backup for', name)

## A — Baseline: PixelCNN++ (20 epoch)

In [ ]:
%cd /content/COMP547PROJECT
!git pull origin main
!PYTHONPATH=src python scripts/train_pixelcnnpp.py \
  --dataset fashion_mnist \
  --epochs 20 \
  --batch-size 16 \
  --num-workers 0 \
  --save-every-epochs 5 \
  --save-dir results/pixelcnnpp_fashion_e20

In [ ]:
from common.checkpointing import backup_results_tree
p = backup_results_tree('results/pixelcnnpp_fashion_e20', '/content/drive/MyDrive/comp547_outputs/fashion_mnist', 'auto')
print('Drive backup:', p)

In [ ]:
from pathlib import Path
ckpt_dir = Path('results/pixelcnnpp_fashion_e20/checkpoints')
for name in ['last.pt', 'best.pt']:
    print(name, (ckpt_dir / name).exists())

In [ ]:
from pathlib import Path
ckpt_dir = Path('results/pixelcnnpp_fashion_e20/checkpoints')
ckpt = ckpt_dir / 'best.pt'
if not ckpt.exists():
    ckpt = sorted(ckpt_dir.glob('*.pt'))[-1]
print('Using', ckpt)
!PYTHONPATH=src python scripts/eval_pixelcnnpp.py \
  --checkpoint "{ckpt}" \
  --out-json results/pixelcnnpp_fashion_e20/eval/fashion_eval.json \
  --out-grid results/pixelcnnpp_fashion_e20/eval/fashion_grid.png \
  --sample-batch-size 25

## B — ARPG (20 epoch)

In [ ]:
%cd /content/COMP547PROJECT
!PYTHONPATH=src python scripts/train_arpg.py \
  --dataset fashion_mnist \
  --data-dir data \
  --save-dir results/arpg_fashion \
  --epochs 20 \
  --batch-size 16 \
  --d-model 192 \
  --n-heads 6 \
  --n-layers 6 \
  --num-workers 0 \
  --save-every-epochs 5 \
  --seed 1

In [ ]:
from common.checkpointing import backup_results_tree
p = backup_results_tree('results/arpg_fashion', '/content/drive/MyDrive/comp547_outputs/fashion_mnist', 'auto')
print('Drive backup:', p)

In [ ]:
from pathlib import Path
ckpt_dir = Path('results/arpg_fashion/checkpoints')
ckpt = ckpt_dir / 'best.pt'
if not ckpt.exists():
    ckpt = sorted(ckpt_dir.glob('*.pt'))[-1]
print('ARPG checkpoint:', ckpt)
!PYTHONPATH=src python scripts/eval_arpg.py \
  --checkpoint "{ckpt}" \
  --out-dir results/arpg_fashion/eval \
  --ks 1,2,4,7,14,28,56,112,196,392,784 \
  --schedules random,raster,row \
  --n-samples 25 \
  --seed 42 \
  --top-p 0.9 \
  --temperature 1.0

## C — Sunum / Rapor çıktıları

In [ ]:
import glob
metrics = sorted(glob.glob('results/pixelcnnpp_fashion_e20/metrics/*.json'))
metrics_path = metrics[-1] if metrics else ''
!PYTHONPATH=src python scripts/build_fashion_report.py \
  --baseline-eval-json results/pixelcnnpp_fashion_e20/eval/fashion_eval.json \
  --arpg-sweep-json results/arpg_fashion/eval/sweep.json \
  --baseline-metrics-json "{metrics_path}" \
  --out-dir results/fashion_presentation

In [ ]:
from IPython.display import Image, display
import json
display(Image('results/fashion_presentation/tradeoff_speed.png'))
print(json.dumps(json.load(open('results/fashion_presentation/presentation_summary.json')), indent=2))

In [ ]:
from common.checkpointing import backup_results_tree
from pathlib import Path
import shutil
root = Path('/content/drive/MyDrive/comp547_outputs/fashion_mnist/final')
root.mkdir(parents=True, exist_ok=True)
for d in ['pixelcnnpp_fashion_e20', 'arpg_fashion', 'fashion_presentation']:
    if Path('results', d).exists():
        dst = root / d
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(f'results/{d}', dst)
print('Final Drive backup OK:', root)